# REI Manuscript — AI Reliability in Qualitative Tourism Research
### Full analysis pipeline (fresh rebuild, 2026-04-15)

This notebook regenerates every CSV, every summary table, and every figure reported in the manuscript from the raw `Comparison_AI&JORT_Dec2_2025_v3.xlsx` workbook and the curated `synonym_groups.csv` lexicon.

Sections:
0. Imports, paths, provenance
1. Merge-aware reader
2. Text normalization
3. Non-compliance taxonomy
4. Tokenization + synonym lexicon
5. Similarity kernel (Lex / Syn / TF-IDF triple)
6-9. Per-sheet parsers (Unlimited, Three Themes, Scholar Lens, Drift)
10. Write records CSVs
11. JORT similarity
12. Cross-model agreement
12b. **Within-model same-mode agreement (Drift R0 replicates)** *(NEW)*
12c. **Three-tier reproducibility ladder** *(NEW)*
13. Scholar distinctiveness
14. Drift analysis (round-by-round, slopes, cycling)
15. Non-compliance table
16. Thematic breadth per response
17. Verbatim scholar vocabulary extraction
18. Figures (Figure 1, Figure 2)
19. Audit log flush


## 0 · Imports, paths, provenance

In [ ]:
from __future__ import annotations
import csv, re, json, datetime, pathlib, itertools, math, os, sys
from collections import Counter, defaultdict
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter

XLSX_NAME = 'Comparison_AI&JORT_Dec2_2025_v3.xlsx'
SYN_NAME  = 'synonym_groups.csv'

def _nb_dir() -> pathlib.Path:
    try:
        return pathlib.Path(__file__).resolve().parent  # type: ignore[name-defined]
    except NameError:
        return pathlib.Path.cwd().resolve()

def _readable(p: pathlib.Path) -> bool:
    # is_file() can return True for phantom tombstones on some filesystems; actually try to open.
    try:
        with p.open('rb') as fh: fh.read(1)
        return True
    except OSError:
        return False

def _find_file(name: str, anchors: list[pathlib.Path], max_up: int = 4) -> pathlib.Path | None:
    seen: set[pathlib.Path] = set()
    queue: list[pathlib.Path] = []
    for a in anchors:
        p = a.resolve()
        queue.append(p)
        for _ in range(max_up):
            if p.parent == p: break
            p = p.parent
            queue.append(p)
    for root in queue:
        if not root.exists() or root in seen: continue
        seen.add(root)
        # look directly + 1 level + 2 levels deep
        for pattern in (name, f'*/{name}', f'*/*/{name}'):
            for m in root.glob(pattern):
                if m.is_file() and _readable(m): return m.resolve()
    return None

_override = os.environ.get('REI_ROOT')
_anchors: list[pathlib.Path] = []
if _override:
    _anchors.append(pathlib.Path(_override).expanduser().resolve())
_anchors += [
    _nb_dir(),
    pathlib.Path.cwd().resolve(),
    pathlib.Path('/sessions/hopeful-funny-edison/mnt/REI Manuscript'),
    pathlib.Path('/sessions/hopeful-funny-edison/mnt'),
]

XLSX = _find_file(XLSX_NAME, _anchors)
if XLSX is None:
    raise FileNotFoundError(
        f'Could not locate {XLSX_NAME}. '
        f'Put it next to this notebook, or set REI_ROOT to its folder.'
    )

# Prefer a "8. Results/" folder that already contains synonym_groups.csv.
SYN_FOUND = _find_file(SYN_NAME, _anchors + [XLSX.parent])
if SYN_FOUND is None:
    raise FileNotFoundError(
        f'Could not locate {SYN_NAME}. '
        f'Expected inside a "8. Results/" folder beside the notebook.'
    )
OUT = SYN_FOUND.parent
SYN = SYN_FOUND
AUDIT_PATH = OUT / '_audit_log.txt'
ROOT = OUT.parent  # usually the "REI Manuscript" folder

print(f'NB_DIR = {_nb_dir()}')
print(f'XLSX   = {XLSX}')
print(f'OUT    = {OUT}')

assert XLSX.exists(), f'missing workbook {XLSX}'
assert SYN.exists(),  f'missing synonym lexicon {SYN}'

_audit_buf: list[str] = []
def audit(msg: str) -> None:
    ts = datetime.datetime.now().strftime('%H:%M:%S')
    line = f'[{ts}] {msg}'
    _audit_buf.append(line)
    print(line)

def flush_audit() -> None:
    header = f'# _audit_log.txt generated {datetime.datetime.now().isoformat()}\n'
    AUDIT_PATH.write_text(header + '\n'.join(_audit_buf) + '\n', encoding='utf-8')

def write_csv(path: pathlib.Path, rows: list[dict], header: list[str] | None = None) -> None:
    if not rows:
        audit(f'  (skipping write: {path.name} empty)'); return
    if header is None: header = list(rows[0].keys())
    with path.open('w', encoding='utf-8', newline='') as fh:
        fh.write(f'# generated {datetime.datetime.now().isoformat()}\n')
        w = csv.DictWriter(fh, fieldnames=header, extrasaction='ignore')
        w.writeheader(); w.writerows(rows)
    audit(f'  wrote {path.name}: {len(rows)} rows')


NB_DIR = /sessions/hopeful-funny-edison/mnt/REI Manuscript/4. Jupyter notebook
XLSX   = /sessions/hopeful-funny-edison/mnt/REI Manuscript/2. Experiment/Comparison_AI&JORT_Dec2_2025_v3.xlsx
OUT    = /sessions/hopeful-funny-edison/mnt/REI Manuscript/8. Results


## 1 · Merge-aware reader

In [2]:
def read_sheet(ws) -> dict[tuple[int,int], str | None]:
    vals: dict[tuple[int,int], str | None] = {}
    for row in ws.iter_rows():
        for c in row:
            vals[(c.row, c.column)] = c.value
    for mr in ws.merged_cells.ranges:
        tl = ws.cell(mr.min_row, mr.min_col).value
        for r in range(mr.min_row, mr.max_row + 1):
            for c in range(mr.min_col, mr.max_col + 1):
                vals[(r, c)] = tl
    return vals


## 2 · Text normalization

In [3]:
TIME_PREFIX_RE = re.compile(
    r'^\s*\(?\s*(?:[0-9]+\s*(?:m|min|mins|minutes))?\s*[0-9]*\s*s(?:ec|econds)?\s*\)?[\s,:\.–\-⏎\n]*',
    re.IGNORECASE)
PAREN_TIME_RE = re.compile(r'^\s*\(Longer\s+time:\s*[^)]+\)\s*[\s⏎\n]*', re.IGNORECASE)
BULLET_RE = re.compile(r'^\s*\d+\s*[\.,\)]\s*', re.MULTILINE)
THEME_HEADER_RE = re.compile(r'^Theme\s*#?\s*\d+\s*[:\.]\s*', re.IGNORECASE | re.MULTILINE)

def scrub_time_prefix(s):
    if not isinstance(s, str): return s, None
    original_head = s[:40]; note = None
    m = PAREN_TIME_RE.match(s)
    if m: note = m.group(0).strip(); s = s[m.end():]
    else:
        m = TIME_PREFIX_RE.match(s[:30])
        if m and m.group(0).strip():
            note = m.group(0).strip(' ⏎\n,:.–-()')
            s = s[m.end():]
    if note: audit(f'  scrubbed time_prefix={note!r} from head={original_head!r}')
    return s, note

def normalize(text):
    if text is None: return ''
    s = str(text).replace('\r\n','\n').replace('\r','\n').replace('⏎','\n')
    return s.strip()

def parse_themes(raw):
    s = normalize(raw)
    s, _ = scrub_time_prefix(s)
    s = THEME_HEADER_RE.sub('', s)
    lines = [BULLET_RE.sub('', ln).strip() for ln in s.split('\n')]
    return [ln for ln in lines if ln and len(ln) < 250][:10]


## 3 · Non-compliance taxonomy (Drift Experiment)

In [4]:
def classify_compliance(raw):
    if raw is None: return False, 'empty'
    s = str(raw).strip()
    if s == '' or s.upper() == 'N/A': return False, 'rollover_NA'
    low = s.lower()
    if low.startswith('i apologize') or 'apologize' in low[:60]:
        return False, 'apology'
    if 'research report' in low[:80]:
        return False, 'report_form'
    if low.startswith(('i appreciate', 'gotcha', 'what you should')):
        return False, 'meta_commentary'
    if low.startswith('i clearly need') or low.startswith("i'm clearly not"):
        return False, 'apology_chain'
    return (len(parse_themes(s)) >= 2, 'ok' if len(parse_themes(s)) >= 2 else 'no_themes_extracted')


## 4 · Tokenization and synonym lexicon

In [5]:
STOPWORDS = {
    'the','a','an','and','or','of','to','in','for','on','with','as','at','by',
    'is','are','was','were','be','been','being','that','this','these','those',
    'it','its','their','our','we','i','you','they','he','she','his','her',
    'from','into','about','across','between','through','during','up','down',
    'not','no','more','less','most','least','all','any','some','each','every',
    'but','so','if','then','than','also','just','only','very','too','can',
    'will','would','should','could','may','might','do','does','did','has','have','had',
    'theme','themes','themed'}
TOKEN_RE = re.compile(r'[a-zA-Z]+')

def tokens(text):
    return [t.lower() for t in TOKEN_RE.findall(text) if t.lower() not in STOPWORDS and len(t) > 1]

def token_set(text): return frozenset(tokens(text))

def load_synonyms(path=SYN):
    mapping = {}
    with path.open(encoding='utf-8') as fh:
        for row in csv.DictReader(fh):
            group = row['group'].strip()
            for w in row['words'].split(','):
                w = w.strip().lower()
                if w: mapping[w] = group
    return mapping

def synonym_tokens(text, syn):
    return frozenset(syn.get(t, t) for t in tokens(text))

syn = load_synonyms()
audit(f'synonym lexicon: {len(set(syn.values()))} groups, {len(syn)} surface terms')


[17:54:59] synonym lexicon: 33 groups, 466 surface terms


## 5 · Similarity kernel

In [6]:
def jaccard_A(a, b):
    if not a and not b: return 1.0
    if not a or not b: return 0.0
    return len(a & b) / len(a | b)

def jaccard_B(a, b):
    if len(a)==0 and len(b)==0: return 1.0
    inter = sum(1 for x in a if x in b)
    union = len(a) + len(b) - inter
    return 0.0 if union==0 else inter/union

def char_ngrams(s, n=3):
    s = ' ' + re.sub(r'\s+', ' ', s.lower().strip()) + ' '
    return Counter(s[i:i+n] for i in range(len(s) - n + 1))

def cosine_A(ca, cb):
    if not ca or not cb: return 0.0
    dot = sum(ca[k]*cb.get(k,0) for k in ca)
    na = math.sqrt(sum(v*v for v in ca.values()))
    nb = math.sqrt(sum(v*v for v in cb.values()))
    return 0.0 if na*nb==0 else dot/(na*nb)

def cosine_B(ca, cb):
    if len(ca)==0 or len(cb)==0: return 0.0
    shared = set(ca) & set(cb)
    dot = sum(ca[k]*cb[k] for k in shared)
    d = math.sqrt(sum(v*v for v in ca.values())) * math.sqrt(sum(v*v for v in cb.values()))
    return 0.0 if d==0 else dot/d

def triple_similarity(a_text, b_text, syn):
    ta, tb = token_set(a_text), token_set(b_text)
    sa, sb = synonym_tokens(a_text, syn), synonym_tokens(b_text, syn)
    ja_A, ja_B = jaccard_A(ta, tb), jaccard_B(ta, tb)
    js_A, js_B = jaccard_A(sa, sb), jaccard_B(sa, sb)
    ca, cb = char_ngrams(a_text), char_ngrams(b_text)
    co_A, co_B = cosine_A(ca, cb), cosine_B(ca, cb)
    assert abs(ja_A - ja_B) < 1e-9
    assert abs(js_A - js_B) < 1e-9
    assert abs(co_A - co_B) < 1e-9
    return {'lex_jaccard': ja_A, 'syn_jaccard': js_A, 'tfidf_cos': co_A}

JORT_THEMES = {
    'T1_immersive':   'Immersive experiences',
    'T2_identity':    'Identity reinforcement',
    'T3_reflection':  'Meaningful reflection',
}

def jort_similarity_block(output_text, syn):
    per = {k: triple_similarity(output_text, v, syn) for k, v in JORT_THEMES.items()}
    rec = {}
    for m in ('lex_jaccard','syn_jaccard','tfidf_cos'):
        vals = [per[k][m] for k in JORT_THEMES]
        rec[f'{m}_max']  = max(vals)
        rec[f'{m}_mean'] = sum(vals)/len(vals)
        for k in JORT_THEMES: rec[f'{m}_{k}'] = per[k][m]
    return rec


## 6 · Parse `Unlimited Themes` (14 records, 7 prompt groups)

In [7]:
def parse_unlimited(wb):
    ws = wb['Unlimited Themes']
    assert ws.max_column == 5
    vals = read_sheet(ws)
    prompt_ranges = sorted((mr.min_row, mr.max_row) for mr in ws.merged_cells.ranges if mr.min_col == 4 == mr.max_col)
    prompt_id_of_row = {r: f'P{i+1}' for i, (r0, r1) in enumerate(prompt_ranges) for r in range(r0, r1+1)}
    records = []
    for r in range(4, 18):
        no, model, focus, prompt, output = (vals.get((r, c)) for c in range(1, 6))
        if model is None:
            audit(f'  Unlimited R{r}: skipped (no model)'); continue
        records.append({
            'sheet':'unlimited','row':r,'no':no,
            'model':str(model).strip(),
            'focus':str(focus).strip() if focus else '',
            'prompt_id':prompt_id_of_row.get(r,'?'),
            'prompt_text':str(prompt).strip() if prompt else '',
            'output_raw':normalize(output),
            'output_norm':normalize(output).lower(),
        })
    audit(f'Unlimited Themes: {len(records)} records; {len(prompt_ranges)} prompt groups')
    assert len(records) == 14 and len(prompt_ranges) == 7
    return records

wb = load_workbook(XLSX, data_only=True)
audit(f'workbook sheets: {wb.sheetnames}')
rec_u = parse_unlimited(wb)
rec_u[0]


[17:54:59] workbook sheets: ['Unlimited Themes', '3 Themes', 'Scholar Lens', 'Drift Experiment']
[17:54:59] Unlimited Themes: 14 records; 7 prompt groups


{'sheet': 'unlimited',
 'row': 4,
 'no': 1,
 'model': 'ChatGPT 5 (Auto)',
 'focus': 'Basic',
 'prompt_id': 'P1',
 'prompt_text': 'Perform coding across the uploaded transcripts and carry out a thematic analysis.',
 'output_raw': 'Theme #1: Travel as Escape and Renewal\nTheme #2: Connection and Community\nTheme #3: Learning and Personal Growth\nTheme #4: Environmental and Ethical Awareness\nTheme #5: Authenticity and Local Engagement\nTheme #6: Transformative Impacts Beyond the Trip',
 'output_norm': 'theme #1: travel as escape and renewal\ntheme #2: connection and community\ntheme #3: learning and personal growth\ntheme #4: environmental and ethical awareness\ntheme #5: authenticity and local engagement\ntheme #6: transformative impacts beyond the trip'}

## 7 · Parse `3 Themes` (4 records)

In [8]:
def parse_three(wb):
    ws = wb['3 Themes']
    assert ws.max_row == 6 and ws.max_column == 5
    vals = read_sheet(ws)
    recs = []
    for r in range(3, 7):
        no, model, focus, prompt, output = (vals.get((r, c)) for c in range(1, 6))
        recs.append({
            'sheet':'three_themes','row':r,'no':no,
            'model':str(model).strip().replace('\n',' '),
            'focus':str(focus).strip() if focus else '',
            'prompt_text':str(prompt).strip() if prompt else '',
            'output_raw':normalize(output),
            'output_norm':normalize(output).lower(),
        })
    audit(f'3 Themes: {len(recs)} records')
    assert len(recs) == 4
    return recs

rec_t = parse_three(wb)
[r['model'] for r in rec_t]


[17:54:59] 3 Themes: 4 records


['ChatGPT 5 (Auto)',
 'ChatGPT 5 (Thinking + Study)',
 'Claude (Sonnet 4)',
 'Claude (Sonnet 4) Reaserch + Extended Thinking']

## 8 · Parse `Scholar Lens` (24 records = 4 models × 6 scholars)

In [9]:
SCHOLAR_ORDER = ['Hunt','Pan','Mowen','Fletcher','Escobar','Crompton']
SCHOLAR_COLS = [(4,5),(6,7),(8,9),(10,11),(12,13),(14,15)]

def parse_scholar(wb):
    ws = wb['Scholar Lens']
    assert ws.max_row == 6 and ws.max_column == 15
    vals = read_sheet(ws)
    recs = []
    for r in range(3, 7):
        no, model, focus = vals.get((r,1)), vals.get((r,2)), vals.get((r,3))
        for sch, (pc, oc) in zip(SCHOLAR_ORDER, SCHOLAR_COLS):
            prompt = vals.get((r, pc)); output = vals.get((r, oc))
            if prompt and sch.lower() not in str(prompt).lower():
                audit(f'  WARNING: scholar {sch} not in prompt at R{r}')
            recs.append({
                'sheet':'scholar','row':r,'no':no,
                'model':str(model).strip().replace('\n',' '),
                'focus':str(focus).strip() if focus else '',
                'scholar':sch,
                'prompt_text':str(prompt).strip() if prompt else '',
                'output_raw':normalize(output),
                'output_norm':normalize(output).lower(),
            })
    audit(f'Scholar Lens: {len(recs)} records')
    assert len(recs) == 24
    return recs

rec_s = parse_scholar(wb)
Counter((r['model'], r['scholar']) for r in rec_s).most_common(3)


[17:54:59] Scholar Lens: 24 records


[(('ChatGPT 5 (Auto)', 'Hunt'), 1),
 (('ChatGPT 5 (Auto)', 'Pan'), 1),
 (('ChatGPT 5 (Auto)', 'Mowen'), 1)]

## 9 · Parse `Drift Experiment` (220 records = 4 models × 5 tones × 11 rounds)

In [10]:
TONES = ['Supportive','Mild Rejection','Strong Rejection','With Lens (Carter)','With Lens (Bing)']
DRIFT_BLOCKS = [
    ('ChatGPT 5.1 (Auto)', 3, 7),
    ('Claude Sonnet 4.5', 8, 12),
    ('ChatGPT 5.1 (Thinking + Study)', 13, 17),
    ('Claude Sonnet 4.5 (Research + Extended Thinking)', 18, 22),
]

def parse_drift(wb):
    ws = wb['Drift Experiment']
    assert ws.max_row == 22 and ws.max_column == 17
    vals = read_sheet(ws)
    recs = []
    for model_name, r0, r1 in DRIFT_BLOCKS:
        for i, r in enumerate(range(r0, r1 + 1)):
            tone = TONES[i]
            prompt_baseline = vals.get((r, 3))
            followup        = vals.get((r, 6))
            for round_idx, col in enumerate([4] + list(range(7, 17))):
                raw = vals.get((r, col))
                raw_s = '' if raw is None else str(raw)
                scrubbed, time_note = scrub_time_prefix(raw_s)
                is_compliant, reason = classify_compliance(scrubbed)
                recs.append({
                    'sheet':'drift','row':r,
                    'model':model_name,'tone':tone,'round':round_idx,
                    'prompt_baseline':str(prompt_baseline).strip() if prompt_baseline else '',
                    'followup_text':str(followup).strip() if followup else '',
                    'output_raw':normalize(raw),
                    'output_norm':normalize(scrubbed).lower() if is_compliant else '',
                    'compliant':is_compliant,'reason_code':reason,
                    'time_note':time_note or '',
                })
            spill = vals.get((r, 17))
            if spill: audit(f'  Drift R{r} col Q spillover: {str(spill)[:60]!r}')
    n_comp = sum(1 for x in recs if x['compliant'])
    audit(f'Drift Experiment: {len(recs)} records ({n_comp} compliant, {len(recs)-n_comp} non-compliant)')
    assert len(recs) == 220
    return recs

rec_d = parse_drift(wb)
Counter(r['reason_code'] for r in rec_d if not r['compliant'])


[17:54:59]   scrubbed time_prefix='1m50s' from head='1m50s\nGotcha — my lists clearly aren’t m'
[17:54:59]   Drift R18 col Q spillover: 'What You Should Do Now\n1. Select your working themes based o'
[17:54:59]   scrubbed time_prefix='27s' from head='27s\nI apologize for misunderstanding. Be'
[17:54:59] Drift Experiment: 220 records (205 compliant, 15 non-compliant)


Counter({'rollover_NA': 8,
         'meta_commentary': 2,
         'apology': 2,
         'report_form': 2,
         'apology_chain': 1})

## 10 · Write `records_*.csv`

In [11]:
write_csv(OUT/'records_unlimited.csv',    rec_u)
write_csv(OUT/'records_three_themes.csv', rec_t)
write_csv(OUT/'records_scholar.csv',      rec_s)
write_csv(OUT/'records_drift.csv',        rec_d)


[17:54:59]   wrote records_unlimited.csv: 14 rows
[17:54:59]   wrote records_three_themes.csv: 4 rows
[17:54:59]   wrote records_scholar.csv: 24 rows


[17:54:59]   wrote records_drift.csv: 220 rows


## 11 · JORT similarity

In [12]:
def compute_jort_similarity(records, syn, tag):
    out = []
    for rec in records:
        text = rec.get('output_norm') or rec.get('output_raw') or ''
        if not text: continue
        row = {'source':tag,'row':rec.get('row'),'model':rec.get('model'),
               'focus':rec.get('focus',''),'scholar':rec.get('scholar',''),
               'tone':rec.get('tone',''),'round':rec.get('round',''),
               'prompt_id':rec.get('prompt_id','')}
        row.update(jort_similarity_block(text, syn))
        out.append(row)
    return out

jort_rows = []
jort_rows += compute_jort_similarity(rec_u, syn, 'unlimited')
jort_rows += compute_jort_similarity(rec_t, syn, 'three_themes')
jort_rows += compute_jort_similarity(rec_s, syn, 'scholar')
write_csv(OUT/'jort_similarity.csv', jort_rows)


[17:54:59]   wrote jort_similarity.csv: 42 rows


## 12 · Cross-model agreement

In [13]:
def compute_cross_model_agreement(records, group_key, syn):
    groups = defaultdict(list)
    for rec in records:
        text = rec.get('output_norm') or rec.get('output_raw') or ''
        if not text: continue
        groups[group_key(rec)].append(rec)
    rows = []
    for gk, recs in groups.items():
        for a, b in itertools.combinations(recs, 2):
            sim = triple_similarity(a['output_norm'], b['output_norm'], syn)
            rows.append({'group':str(gk),'model_a':a['model'],'model_b':b['model'],
                         'row_a':a['row'],'row_b':b['row'], **sim})
    return rows

cma  = compute_cross_model_agreement(rec_u, lambda r: r['prompt_id'], syn)
cmat = compute_cross_model_agreement(rec_t, lambda r: r['focus'],     syn)
cmas = compute_cross_model_agreement(rec_s, lambda r: r['scholar'],   syn)
for r in cma:  r['sheet'] = 'unlimited'
for r in cmat: r['sheet'] = 'three_themes'
for r in cmas: r['sheet'] = 'scholar'
audit(f'  unlimited: {len(cma)}; three_themes: {len(cmat)}; scholar: {len(cmas)}')
assert len(cma)==7 and len(cmat)==6 and len(cmas)==36
write_csv(OUT/'cross_model_agreement.csv', cma + cmat + cmas)


[17:54:59]   unlimited: 7; three_themes: 6; scholar: 36
[17:54:59]   wrote cross_model_agreement.csv: 49 rows


## § 12b · Within-model same-mode consistency (Drift R0 replicates)

For each of the four model–mode cells, the Drift Experiment generated **five independent Round-0 responses** — one per tone condition, produced *before* any tone was applied. These five responses share the identical *Three Themes* prompt and therefore measure each model's intrinsic run-to-run consistency.

We compute the same Lex./Syn./TF-IDF triple on all C(5,2)=10 within-cell pairs, yielding **4 × 10 = 40 pairs**. This serves as the within-model baseline layer in the three-tier reproducibility ladder reported in §4.2 of the manuscript (within-model same-mode → within-family cross-mode → cross-family).

Output: `8. Results/within_model_agreement.csv`


In [14]:
# ---- Within-model same-mode agreement from Drift Round-0 replicates -----
# 4 models × C(5,2)=10 pairs per model = 40 pairs
r0 = [r for r in rec_d if r.get('round') == 0]
by_model = defaultdict(list)
for r in r0:
    by_model[r['model']].append(r)

wm_rows = []
for m, rows in by_model.items():
    for a, b in itertools.combinations(rows, 2):
        sim = triple_similarity(a['output_norm'], b['output_norm'], syn)
        wm_rows.append({
            'sheet':       'within_model',
            'model':       m,
            'tone_a':      a['tone'],
            'tone_b':      b['tone'],
            'round_a':     0,
            'round_b':     0,
            **{k: round(v, 6) for k, v in sim.items()},
        })

audit(f'within-model same-mode pairs: {len(wm_rows)}')
assert len(wm_rows) == 40, f'expected 40 pairs, got {len(wm_rows)}'
write_csv(OUT/'within_model_agreement.csv', wm_rows)

# summary
from statistics import mean, stdev
print('Per-model within-model same-mode consistency (n=10 pairs each):')
print(f"{'Model':55s} {'Lex':>8s} {'Syn':>8s} {'TFIDF':>8s}")
for m in sorted(by_model):
    my = [r for r in wm_rows if r['model'] == m]
    print(f"{m:55s} "
          f"{mean(r['lex_jaccard'] for r in my):>8.3f} "
          f"{mean(r['syn_jaccard'] for r in my):>8.3f} "
          f"{mean(r['tfidf_cos']   for r in my):>8.3f}")
print()
print(f"Overall (n={len(wm_rows)}):")
print(f"  Lex   mean={mean(r['lex_jaccard'] for r in wm_rows):.4f}  "
      f"SD={stdev(r['lex_jaccard'] for r in wm_rows):.4f}")
print(f"  Syn   mean={mean(r['syn_jaccard'] for r in wm_rows):.4f}  "
      f"SD={stdev(r['syn_jaccard'] for r in wm_rows):.4f}")
print(f"  TFIDF mean={mean(r['tfidf_cos']   for r in wm_rows):.4f}  "
      f"SD={stdev(r['tfidf_cos']   for r in wm_rows):.4f}")


[17:54:59] within-model same-mode pairs: 40
[17:54:59]   wrote within_model_agreement.csv: 40 rows
Per-model within-model same-mode consistency (n=10 pairs each):
Model                                                        Lex      Syn    TFIDF
ChatGPT 5.1 (Auto)                                         0.714    0.848    0.844
ChatGPT 5.1 (Thinking + Study)                             0.634    0.757    0.811
Claude Sonnet 4.5                                          0.829    1.000    0.925
Claude Sonnet 4.5 (Research + Extended Thinking)           0.564    0.636    0.807

Overall (n=40):
  Lex   mean=0.6854  SD=0.2440
  Syn   mean=0.8101  SD=0.1891
  TFIDF mean=0.8465  SD=0.1233


## § 12c · Three-tier reproducibility ladder (Three Themes prompt)

Holding the *Three Themes* prompt constant, agreement falls in three tiers:

1. **Within-model same-mode** (same AI, same mode, same prompt, repeated): n = 40 from `within_model_agreement.csv`
2. **Within-family cross-mode** (same AI, different modes): n = 2 — ChatGPT Auto vs. ChatGPT T+S, Claude Sonnet vs. Claude R+ET
3. **Cross-family** (ChatGPT vs. Claude): n = 4

This cell re-computes the ladder from the two CSVs so the manuscript values can be regenerated programmatically.


In [15]:
# ---- Three-tier reproducibility ladder on the Three Themes prompt -------
from statistics import mean

# tier 1: within-model same-mode (all 40 pairs)
tier1 = [r['syn_jaccard'] for r in wm_rows]

# tier 2/3: cross-model pairs on the Three Themes sheet
three_pairs = [r for r in (cma + cmat + cmas) if r['sheet'] == 'three_themes']
def family(m):
    return 'ChatGPT' if 'ChatGPT' in m else 'Claude' if 'Claude' in m else m

tier2 = [r['syn_jaccard'] for r in three_pairs if family(r['model_a']) == family(r['model_b'])]
tier3 = [r['syn_jaccard'] for r in three_pairs if family(r['model_a']) != family(r['model_b'])]

print('Three-tier reproducibility ladder (Syn-Aware Jaccard, Three Themes prompt):')
print(f"  Tier 1 | Within-model same-mode   n={len(tier1):2d}  mean={mean(tier1):.3f}")
print(f"  Tier 2 | Within-family cross-mode n={len(tier2):2d}  mean={mean(tier2):.3f}")
print(f"  Tier 3 | Cross-family             n={len(tier3):2d}  mean={mean(tier3):.3f}")
print()
print(f"  Largest drop: Tier 1 → Tier 2 = {mean(tier1) - mean(tier2):+.3f}")
print(f"                Tier 2 → Tier 3 = {mean(tier2) - mean(tier3):+.3f}")


Three-tier reproducibility ladder (Syn-Aware Jaccard, Three Themes prompt):
  Tier 1 | Within-model same-mode   n=40  mean=0.810
  Tier 2 | Within-family cross-mode n= 2  mean=0.667
  Tier 3 | Cross-family             n= 4  mean=0.417

  Largest drop: Tier 1 → Tier 2 = +0.143
                Tier 2 → Tier 3 = +0.250


## 13 · Scholar distinctiveness

In [16]:
def compute_scholar_distinctiveness(records, syn):
    by_model = defaultdict(dict)
    for rec in records: by_model[rec['model']][rec['scholar']] = rec
    rows = []
    for model, sd in by_model.items():
        for a, b in itertools.combinations(sd.keys(), 2):
            sim = triple_similarity(sd[a]['output_norm'], sd[b]['output_norm'], syn)
            rows.append({'model':model,'scholar_a':a,'scholar_b':b, **sim})
    return rows

sd = compute_scholar_distinctiveness(rec_s, syn)
assert len(sd) == 60
write_csv(OUT/'scholar_distinctiveness.csv', sd)


[17:54:59]   wrote scholar_distinctiveness.csv: 60 rows

## 14 · Drift analysis

In [17]:
def compute_drift(records, syn):
    groups = defaultdict(list)
    for rec in records:
        groups[(rec['model'], rec['tone'])].append(rec)
    rows = []
    for (model, tone), recs in groups.items():
        recs.sort(key=lambda r: r['round'])
        r0 = recs[0]
        if not r0['compliant']:
            audit(f'  {model}/{tone}: Round 0 non-compliant; anchoring on first compliant round')
            anchors = [r for r in recs if r['compliant']]
            r0 = anchors[0] if anchors else r0
        prev = r0
        for rec in recs:
            if not rec['compliant']:
                rows.append({'model':model,'tone':tone,'round':rec['round'],
                             'compliant':False,'reason':rec['reason_code'],
                             **{f'{m}_{k}':None for m in ('lex_jaccard','syn_jaccard','tfidf_cos')
                                                 for k in ('global','local','jort')}})
                continue
            g = triple_similarity(rec['output_norm'], r0['output_norm'], syn)
            l = triple_similarity(rec['output_norm'], prev['output_norm'], syn)
            j = jort_similarity_block(rec['output_norm'], syn)
            rows.append({'model':model,'tone':tone,'round':rec['round'],
                         'compliant':True,'reason':rec['reason_code'],
                         'lex_jaccard_global':g['lex_jaccard'],
                         'syn_jaccard_global':g['syn_jaccard'],
                         'tfidf_cos_global':  g['tfidf_cos'],
                         'lex_jaccard_local': l['lex_jaccard'],
                         'syn_jaccard_local': l['syn_jaccard'],
                         'tfidf_cos_local':   l['tfidf_cos'],
                         'lex_jaccard_jort':  j['lex_jaccard_mean'],
                         'syn_jaccard_jort':  j['syn_jaccard_mean'],
                         'tfidf_cos_jort':    j['tfidf_cos_mean']})
            prev = rec
    return rows

da = compute_drift(rec_d, syn)
write_csv(OUT/'drift_analysis.csv', da)


[17:54:59]   wrote drift_analysis.csv: 220 rows


## 15 · Non-compliance table

In [18]:
nc = [r for r in rec_d if not r['compliant']]
audit(f'Non-compliance: {len(nc)} cells')
by_reason = Counter(r['reason_code'] for r in nc)
for reason, n in by_reason.most_common(): audit(f'  {reason}: {n}')
write_csv(OUT/'non_compliant_outputs.csv',
    [{'row':r['row'],'model':r['model'],'tone':r['tone'],'round':r['round'],
      'reason':r['reason_code'],'raw_preview':r['output_raw'][:120]} for r in nc])


[17:54:59] Non-compliance: 15 cells
[17:54:59]   rollover_NA: 8
[17:54:59]   meta_commentary: 2
[17:54:59]   apology: 2
[17:54:59]   report_form: 2
[17:54:59]   apology_chain: 1
[17:54:59]   wrote non_compliant_outputs.csv: 15 rows


## 16 · Thematic breadth per response

In [19]:
# ---- Thematic breadth: canonical concepts per response -----------------
def canonical_count(text):
    """Number of unique synonym-collapsed tokens in a response."""
    return len(synonym_tokens(text, syn))

breadth_rows = []
for sheet_label, recs in [("unlimited", rec_u),
                          ("three_themes", rec_t),
                          ("scholar",      rec_s)]:
    for r in recs:
        breadth_rows.append({
            "sheet":   sheet_label,
            "model":   r.get("model", ""),
            "row_id":  r.get("row", r.get("row_id", "")),
            "scholar": r.get("scholar", ""),
            "canonical_concepts": canonical_count(r["output_norm"]),
        })

# Write full per-response table and a one-line summary.
write_csv(OUT / "thematic_breadth.csv", breadth_rows)

from statistics import mean
for sheet in ("unlimited", "three_themes", "scholar"):
    vals = [r["canonical_concepts"] for r in breadth_rows if r["sheet"] == sheet]
    audit(f"  breadth[{sheet}]: n={len(vals)}  mean={mean(vals):.2f}  min={min(vals)}  max={max(vals)}")
print(f"  wrote thematic_breadth.csv: {len(breadth_rows)} rows")

# Print the headline numbers that flow into the manuscript.
mu = mean([r["canonical_concepts"] for r in breadth_rows if r["sheet"] == "unlimited"])
mt = mean([r["canonical_concepts"] for r in breadth_rows if r["sheet"] == "three_themes"])
ms = mean([r["canonical_concepts"] for r in breadth_rows if r["sheet"] == "scholar"])
print(f"\n  Mean canonical concepts per response:")
print(f"    Unlimited Themes : {mu:.2f}")
print(f"    Three Themes     : {mt:.2f}")
print(f"    Scholar Lens     : {ms:.2f}")
print(f"    Fold reduction (Unlimited / Three) : {mu/mt:.2f}x")


[17:54:59]   wrote thematic_breadth.csv: 42 rows
[17:54:59]   breadth[unlimited]: n=14  mean=20.64  min=13  max=35
[17:54:59]   breadth[three_themes]: n=4  mean=6.00  min=6  max=6
[17:54:59]   breadth[scholar]: n=24  mean=5.92  min=3  max=7
  wrote thematic_breadth.csv: 42 rows

  Mean canonical concepts per response:
    Unlimited Themes : 20.64
    Three Themes     : 6.00
    Scholar Lens     : 5.92
    Fold reduction (Unlimited / Three) : 3.44x


## 17 · Verbatim scholar vocabulary extraction

In [20]:
import pandas as pd
import re
from collections import Counter
from pathlib import Path

RESULTS = Path('/sessions/hopeful-funny-edison/mnt/REI Manuscript/8. Results')
sc = pd.read_csv(RESULTS / 'records_scholar.csv', comment='#')

# Each output_raw string looks like 'Theme #1: Phrase\nTheme #2: Phrase\nTheme #3: Phrase'.
# Parse the phrase after each 'Theme #N:' marker, preserving verbatim capitalisation.
THEME_RE = re.compile(r'Theme\s*#?\s*\d+\s*[:\-\.]\s*(.+?)(?=(?:\s*\n+Theme\s*#?\s*\d+)|\s*$)', re.IGNORECASE | re.DOTALL)

def parse_themes(text):
    if not isinstance(text, str):
        return []
    found = [m.strip().rstrip('.;,') for m in THEME_RE.findall(text)]
    # Fallback: split on lines if regex missed (robust to stray formatting)
    if len(found) < 3:
        lines = [ln.strip() for ln in text.split('\n') if ':' in ln]
        alt = []
        for ln in lines:
            after = ln.split(':', 1)[-1].strip().rstrip('.;,')
            if after:
                alt.append(after)
        if len(alt) >= len(found):
            found = alt
    return found

rows = []
for scholar, grp in sc.groupby('scholar'):
    themes = []
    for _, r in grp.iterrows():
        themes.extend(parse_themes(r['output_raw']))
    cnt = Counter(t.strip() for t in themes)
    ordered = sorted(cnt.items(), key=lambda kv: (-kv[1], kv[0].lower()))
    for rank, (theme, n) in enumerate(ordered, 1):
        rows.append({'scholar': scholar, 'rank': rank, 'theme': theme, 'frequency': n,
                     'total_themes': len(themes), 'unique_themes': len(cnt)})

vocab = pd.DataFrame(rows)
out = RESULTS / 'scholar_vocabulary.csv'
with open(out, 'w', encoding='utf-8') as fh:
    fh.write(f'# generated {pd.Timestamp.utcnow().isoformat()} by REI_Analysis.ipynb \u00a7 17\n')
    vocab.to_csv(fh, index=False)
print(f'Wrote {out} with {len(vocab)} rows')

for scholar in ['Hunt','Mowen','Pan','Crompton','Fletcher','Escobar']:
    v = vocab[vocab['scholar']==scholar]
    sig = [f"{t} \u00d7{n}" if n>1 else t for t,n in zip(v['theme'], v['frequency'])]
    print(f'{scholar:>9s} (total={int(v.iloc[0]["total_themes"])}, unique={int(v.iloc[0]["unique_themes"])}): {", ".join(sig)}')


Wrote /sessions/hopeful-funny-edison/mnt/REI Manuscript/8. Results/scholar_vocabulary.csv with 66 rows
     Hunt (total=12, unique=12): Authentic Connection, Community Connection, Community immersion, Environmental Awakening, Environmental Awareness, Environmental stewardship, Identity Reinforcement, Immersive Experiences, Meaningful Reflection, Personal Transformation, Transformative Impact, Transformative learning
    Mowen (total=12, unique=10): Environmental Consciousness ×2, Environmental Stewardship ×2, Community Connection, Community Engagement, Experiential Connection, Personal Transformation, Physical Activity, Place Attachment, Sustained Transformation, Transformative Experience
      Pan (total=12, unique=11): Community Connection ×2, Authentic Community, Behavior Change, Behavioral Commitment, Behavioral Transformation, Ecological Identity, Environmental Consciousness, Environmental Stewardship, Eudaimonic Awakening, Sustainable Engagement, Transformative Experience
 Crompt

## 18 · Figures for the manuscript

In [21]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

FIG_DIR = OUT / "figures"
FIG_DIR.mkdir(exist_ok=True)

# Match manuscript typography (Times New Roman 12pt body in the docx).
plt.rcParams.update({
    "font.family":      "serif",
    "font.serif":       ["Times New Roman", "DejaVu Serif"],
    "font.size":        11,
    "axes.titlesize":   12,
    "axes.labelsize":   11,
    "xtick.labelsize":  10,
    "ytick.labelsize":  10,
    "legend.fontsize":  9.5,
    "axes.spines.top":   False,
    "axes.spines.right": False,
})
audit(f"Figures will be written to {FIG_DIR}")


[17:55:00] Figures will be written to /sessions/hopeful-funny-edison/mnt/REI Manuscript/8. Results/figures


In [22]:
# ---- Figure 1: Inter-Scholar Thematic Distinctiveness ----------------
sd = pd.read_csv(OUT / "scholar_distinctiveness.csv", skiprows=1)

scholars = ["Hunt", "Pan", "Mowen", "Crompton", "Fletcher", "Escobar"]
mat = np.full((len(scholars), len(scholars)), np.nan)
for i, a in enumerate(scholars):
    for j, b in enumerate(scholars):
        if i == j: continue
        sub = sd[((sd.scholar_a == a) & (sd.scholar_b == b)) |
                 ((sd.scholar_a == b) & (sd.scholar_b == a))]
        if len(sub):
            mat[i, j] = sub["syn_jaccard"].mean()
row_means = np.nanmean(mat, axis=1)

# Wider overall figure + wider right panel so row-mean numeric labels never clip.
fig = plt.figure(figsize=(9.4, 5.6))
gs  = fig.add_gridspec(1, 3, width_ratios=[5.4, 2.2, 0.25], wspace=0.08)
ax       = fig.add_subplot(gs[0, 0])
ax_mean  = fig.add_subplot(gs[0, 1])
ax_cb    = fig.add_subplot(gs[0, 2])

cmap = plt.cm.YlGnBu
im   = ax.imshow(mat, cmap=cmap, vmin=0, vmax=0.5)
for i in range(len(scholars)):
    for j in range(len(scholars)):
        if i == j:
            ax.text(j, i, "\u2014", ha="center", va="center", color="#666", fontsize=10)
        else:
            v = mat[i, j]
            ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                    color="white" if v > 0.27 else "black", fontsize=10)
ax.set_xticks(range(len(scholars)))
ax.set_yticks(range(len(scholars)))
ax.set_xticklabels(scholars, rotation=30, ha="right")
ax.set_yticklabels(scholars)
ax.set_title("Figure 1. Inter-Scholar Thematic Distinctiveness\n"
             "(mean Synonym-Aware Jaccard, pairs across 4 models)", pad=10)

# Side panel: row means, uniform 3-decimal labels, with headroom for text.
y_pos = range(len(scholars))
ax_mean.barh(y_pos, row_means, color="#5a7da4", edgecolor="black", linewidth=0.4)
x_upper = max(0.45, float(np.nanmax(row_means)) + 0.10)
for i, m in enumerate(row_means):
    ax_mean.text(m + 0.008, i, f"{m:.3f}", va="center", fontsize=9.5)
ax_mean.set_xlim(0, x_upper)
ax_mean.set_ylim(len(scholars) - 0.5, -0.5)
ax_mean.set_xticks([0.0, 0.1, 0.2, 0.3, 0.4])
ax_mean.tick_params(axis="y", which="both", left=False, labelleft=False)
ax_mean.set_xlabel("Row mean (Syn. Jaccard)")
ax_mean.spines["left"].set_visible(False)
ax_mean.spines["top"].set_visible(False)
ax_mean.spines["right"].set_visible(False)

cbar = fig.colorbar(im, cax=ax_cb)
cbar.set_label("Mean Syn. Jaccard")

fig1_path = FIG_DIR / "Figure1_ScholarMatrix.png"
fig.savefig(fig1_path, dpi=300, bbox_inches="tight")
plt.close(fig)
audit(f"Figure 1 written: {fig1_path.name}  ({fig1_path.stat().st_size:,} B)")
print(f"  ->  {fig1_path}")


[17:55:01] Figure 1 written: Figure1_ScholarMatrix.png  (288,213 B)
  ->  /sessions/hopeful-funny-edison/mnt/REI Manuscript/8. Results/figures/Figure1_ScholarMatrix.png


In [23]:
# ---- Figure 2: Thematic Drift Trajectories ---------------------------
dr = pd.read_csv(OUT / "drift_analysis.csv", skiprows=1)
dr = dr[dr["compliant"] == True].copy()

def _canon_tone(t):
    s = str(t).strip().lower()
    if "support" in s:  return "Supportive"
    if "mild"    in s:  return "Mild Rejection"
    if "strong"  in s:  return "Strong Rejection"
    if "carter"  in s:  return "Lens (Hunt)"
    if "bing"    in s:  return "Lens (Pan)"
    return t
dr["tone_c"] = dr["tone"].apply(_canon_tone)

models = ["ChatGPT 5.1 (Auto)",
          "ChatGPT 5.1 (Thinking + Study)",
          "Claude Sonnet 4.5",
          "Claude Sonnet 4.5 (Research + Extended Thinking)"]
short_model = {
    "ChatGPT 5.1 (Auto)":                                "ChatGPT 5.1 Auto",
    "ChatGPT 5.1 (Thinking + Study)":                    "ChatGPT 5.1 T+S",
    "Claude Sonnet 4.5":                                 "Claude Sonnet 4.5 (Default)",
    "Claude Sonnet 4.5 (Research + Extended Thinking)":  "Claude Sonnet 4.5 R+ET",
}
tone_order  = ["Supportive", "Mild Rejection", "Strong Rejection",
               "Lens (Hunt)", "Lens (Pan)"]
tone_colors = {"Supportive":       "#3a8c3a",
               "Mild Rejection":   "#d99520",
               "Strong Rejection": "#c44040",
               "Lens (Hunt)":      "#4761ad",
               "Lens (Pan)":        "#8e44ad"}

fig, axes = plt.subplots(2, 2, figsize=(9.4, 6.6), sharex=True, sharey=True)
for ax, model in zip(axes.flat, models):
    sub = dr[dr["model"] == model]
    for tone in tone_order:
        s = sub[sub["tone_c"] == tone].groupby("round")["syn_jaccard_jort"].mean()
        if len(s):
            ax.plot(s.index, s.values, marker="o", markersize=4, linewidth=1.6,
                    color=tone_colors[tone], label=tone, alpha=0.9)
    ax.set_title(short_model[model], fontsize=11)
    ax.set_xticks(range(0, 11))
    ax.set_ylim(-0.005, 0.22)
    ax.grid(True, linestyle=":", alpha=0.45)
    ax.set_axisbelow(True)
    ax.set_xlabel("Round")
    ax.set_ylabel("JORT Syn-Aware Jaccard")

handles = [plt.Line2D([0], [0], color=tone_colors[t], marker="o",
                      linewidth=1.6, label=t) for t in tone_order]
fig.legend(handles=handles, loc="lower center", ncol=5, frameon=False,
           bbox_to_anchor=(0.5, -0.02))
fig.suptitle("Figure 2. Thematic Drift Trajectories Across 10 Rounds of Repeated Prompting",
             fontsize=12, y=1.00)
fig.tight_layout(rect=[0, 0.04, 1, 0.97])

fig2_path = FIG_DIR / "Figure2_Drift.png"
fig.savefig(fig2_path, dpi=300, bbox_inches="tight")
plt.close(fig)
audit(f"Figure 2 written: {fig2_path.name}  ({fig2_path.stat().st_size:,} B)")
print(f"  ->  {fig2_path}")


[17:55:01] Figure 2 written: Figure2_Drift.png  (565,601 B)
  ->  /sessions/hopeful-funny-edison/mnt/REI Manuscript/8. Results/figures/Figure2_Drift.png


## 19 · Flush audit log

In [24]:
flush_audit()
print(f'\nAll artifacts written to {OUT}')
print('Ready for Methods / Results / Findings write-up.')



All artifacts written to /sessions/hopeful-funny-edison/mnt/REI Manuscript/8. Results
Ready for Methods / Results / Findings write-up.
